## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


## 2. Load Dataset

In [ ]:
df = pd.read_csv('retail_sales_dataset.csv')
df.head()


## 3. Dataset Information

In [ ]:
df.info()
df.describe(include='all')


## 4. Missing Values

In [ ]:
print(df.isnull().sum())


## 5. Exploratory Data Analysis

In [ ]:
# Sales by Category
df.groupby('Category')['Sales'].sum().plot(kind='bar', figsize=(6,4), title='Sales by Category')
plt.ylabel('Sales')
plt.show()

# Profit by Region
df.groupby('Region')['Profit'].sum().plot(kind='bar', figsize=(6,4), title='Profit by Region')
plt.show()

# Monthly Sales Trend
monthly=df.copy()
monthly['Order_Date']=pd.to_datetime(monthly['Order_Date'])
monthly.groupby(monthly['Order_Date'].dt.month)['Sales'].sum().plot(marker='o', figsize=(8,4), title='Monthly Sales')
plt.show()

# Quantity Distribution
plt.hist(df['Quantity'], bins=10)
plt.title('Quantity Distribution')
plt.show()

# Sales Distribution
plt.hist(df['Sales'], bins=20)
plt.title('Sales Distribution')
plt.show()


## 6. Feature Engineering

In [ ]:
X=df[['Category','Region','Segment','Quantity','Discount']]
y=df['Sales']


## 7. Train-Test Split

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42)


## 8. Preprocessing

In [ ]:
preprocessor=ColumnTransformer(
    [('cat',OneHotEncoder(handle_unknown='ignore'),
      ['Category','Region','Segment'])],
    remainder='passthrough')


## 9. Model Training

In [ ]:
models={
    'Linear Regression':LinearRegression(),
    'Decision Tree':DecisionTreeRegressor(random_state=42),
    'Random Forest':RandomForestRegressor(random_state=42,n_estimators=100)
}

results=[]

for name,algo in models.items():
    model=Pipeline([
        ('prep',preprocessor),
        ('model',algo)
    ])
    model.fit(X_train,y_train)
    pred=model.predict(X_test)

    results.append({
        'Model':name,
        'R2':r2_score(y_test,pred),
        'MAE':mean_absolute_error(y_test,pred),
        'RMSE':mean_squared_error(y_test,pred)**0.5
    })

results_df=pd.DataFrame(results)
results_df


## 10. Best Model Prediction

In [ ]:
best=Pipeline([
    ('prep',preprocessor),
    ('model',RandomForestRegressor(random_state=42))
])

best.fit(X_train,y_train)
pred=best.predict(X_test)

plt.figure(figsize=(6,6))
plt.scatter(y_test,pred)
plt.xlabel('Actual Sales')
plt.ylabel('Predicted Sales')
plt.title('Actual vs Predicted Sales')
plt.show()


# Conclusion

- Explored the retail dataset using EDA.
- Identified sales trends by category, region, and month.
- Compared three machine learning algorithms.
- Random Forest generally provides better prediction performance than a basic Linear Regression model on this dataset.
- The workflow demonstrates a complete data science pipeline from data loading to model evaluation.
